In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb

In [ ]:
DATA_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/5sec_50%overlap/data"

train_path = os.path.join(DATA_DIR, "DBP_FTEST_train.csv")
val_path   = os.path.join(DATA_DIR, "DBP_FTEST_val.csv")
test_path  = os.path.join(DATA_DIR, "DBP_FTEST_test.csv")

df_train = pd.read_csv(train_path)
df_val   = pd.read_csv(val_path)
df_test  = pd.read_csv(test_path)

print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)

Train: (421829, 16)
Val:   (53254, 16)
Test:  (53482, 16)


In [7]:
print(df_train.head()) 
print(df_train.columns.tolist())

   vpg_skewness  apg_zero_crossing_rate  apg_skewness   ppg_iqr  \
0     -0.496009               -0.903437      2.997517  0.477341   
1     -0.279398                1.215287      2.616810  0.790354   
2     -0.773370               -0.668023      2.806640  0.770992   
3     -0.048958               -0.314903     -1.240632  0.490249   
4     -0.578255                0.391339      2.717168  0.641915   

   ppg_energy_iqr  vpg_shannon_entropy  vpg_kurtosis  apg_shannon_entropy  \
0        1.048839             0.079169     -1.015547            -0.159808   
1        1.247424             0.062313     -0.898346             0.263887   
2        1.227098             0.707743     -1.195279             0.177687   
3        0.925445            -1.486661     -0.437327            -2.299930   
4        1.172104             0.303351     -1.052519             0.223240   

   ppg_variance  vpg_median  vpg_energy_skewness  ppg_energy_variance  \
0      0.006664   -2.948594            -0.182726             

In [ ]:
TARGET = "DBP"

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]

X_val = df_val.drop(columns=[TARGET])
y_val = df_val[TARGET]

X_test = df_test.drop(columns=[TARGET])
y_test = df_test[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:  ", X_val.shape)
print("y_val:  ", y_val.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

X_train: (421829, 15)
y_train: (421829,)
X_val:   (53254, 15)
y_val:   (53254,)
X_test:  (53482, 15)
y_test:  (53482,)


In [ ]:
X_dev = pd.concat([X_train, X_val], axis=0)
y_dev = pd.concat([y_train, y_val], axis=0)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def print_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"RMSE: {rmse:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"R²  : {r2:.4f}")

## SIMPLE LINEAR REGRESSION ##

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(X_dev, y_dev)

y_test_pred = linear_model.predict(X_test)

print("Simple Linear Regression")
print("------------------------")
print_metrics(y_test, y_test_pred)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


## INTERACTION LINEAR REGRESSION ##

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

interaction_model = Pipeline([
    ("poly", PolynomialFeatures(
        degree=2,
        interaction_only=True,
        include_bias=False
    )),
    ("linear", LinearRegression())
])

param_grid = {
    "linear__fit_intercept": [True, False]
}

grid_interaction = GridSearchCV(
    interaction_model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_interaction.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_interaction.best_params_)

y_test_pred = grid_interaction.predict(X_test)

print("\nInteraction Linear Regression")
print("-----------------------------")
print_metrics(y_test, y_test_pred)

## Robust Linear Regression ##

In [ ]:
from sklearn.linear_model import HuberRegressor

robust_model = HuberRegressor(
    max_iter=1000
)

param_grid = {
    "epsilon": [1.1, 1.35, 1.5, 2.0],
    "alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]
}

grid_robust = GridSearchCV(
    robust_model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_robust.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_robust.best_params_)

y_test_pred = grid_robust.predict(X_test)

print("\nRobust Linear Regression")
print("------------------------")
print_metrics(y_test, y_test_pred)

## Fine Decision Tree ##

In [ ]:
from sklearn.tree import DecisionTreeRegressor

fine_tree = DecisionTreeRegressor(
    random_state=42
)

param_grid = {
    "max_depth": [10, 15, 20, 25, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_fine_tree = GridSearchCV(
    fine_tree,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_fine_tree.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_fine_tree.best_params_)

y_test_pred = grid_fine_tree.predict(X_test)

print("\nFine Decision Tree")
print("------------------")
print_metrics(y_test, y_test_pred)

## Random Forest ##

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 1.0]
}

grid_rf = GridSearchCV(
    rf,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_rf.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_rf.best_params_)

y_test_pred = grid_rf.predict(X_test)

print("\nRandom Forest")
print("-------------")
print_metrics(y_test, y_test_pred)

## Linear SVR ##

In [ ]:
from sklearn.svm import SVR

linear_svr = SVR(
    kernel="linear"
)

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "epsilon": [0.01, 0.1, 0.5, 1.0]
}

grid_linear_svr = GridSearchCV(
    linear_svr,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_linear_svr.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_linear_svr.best_params_)

y_test_pred = grid_linear_svr.predict(X_test)

print("\nLinear SVR")
print("----------")
print_metrics(y_test, y_test_pred)

## Extra TREES ##

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

extra_trees = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": [1.0, "sqrt"]
}

grid_extra_trees = GridSearchCV(
    estimator=extra_trees,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_extra_trees.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_extra_trees.best_params_)

y_test_pred = grid_extra_trees.predict(X_test)

print("Extra Trees Regressor")
print("---------------------")
print_metrics(y_test, y_test_pred)

## XGBOOST ##

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    xgb,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_xgb.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_xgb.best_params_)

y_test_pred = grid_xgb.predict(X_test)

print("\nXGBoost")
print("-------")
print_metrics(y_test, y_test_pred)

## Light GBM ##

In [ ]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [15, 31],
    "max_depth": [-1, 10],
    "min_child_samples": [20, 50]
}

grid_lgbm = GridSearchCV(
    lgbm,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_lgbm.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_lgbm.best_params_)

y_test_pred = grid_lgbm.predict(X_test)

print("\nLightGBM")
print("--------")
print_metrics(y_test, y_test_pred)

## Rational Quadratic Gaussian Process Regression ##

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RationalQuadratic,
    ConstantKernel
)

gpr = GaussianProcessRegressor(
    random_state=42,
    normalize_y=True
)

param_grid = {
    "kernel": [
        ConstantKernel(1.0) * RationalQuadratic(
            length_scale=1.0,
            alpha=0.1
        ),
        ConstantKernel(1.0) * RationalQuadratic(
            length_scale=1.0,
            alpha=1.0
        ),
        ConstantKernel(1.0) * RationalQuadratic(
            length_scale=10.0,
            alpha=0.1
        ),
        ConstantKernel(1.0) * RationalQuadratic(
            length_scale=10.0,
            alpha=1.0
        )
    ],
    "alpha": [1e-6, 1e-4, 1e-2]
}

grid_gpr = GridSearchCV(
    gpr,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_gpr.fit(X_dev, y_dev)

print("Best parameters:")
print(grid_gpr.best_params_)

y_test_pred = grid_gpr.predict(X_test)

print("\nRational Quadratic GPR")
print("----------------------")
print_metrics(y_test, y_test_pred)